# Hyperparameter Tuning for Multi-Route Model

This notebook tunes hyperparameters for the multi-route flight delay prediction models.

**Tuning Approach:**
- Use validation set for tuning
- Test set only used for final evaluation
- Manual grid search (not cross-validation) to preserve temporal structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score,
    f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

In [ ]:
# Baseline metrics from notebook 8b (for comparison)
BASELINE_8B = {
    'regression': {
        'Ridge': {'val_r2': 0.4099, 'test_r2': 0.4618, 'params': {'alpha': 100}},
        'Random Forest': {'val_r2': 0.4148, 'test_r2': 0.4858, 
                          'params': {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 5}}
    },
    'classification': {
        'Logistic': {'val_f1': 0.6821, 'test_f1': 0.7062, 'params': {'C': 1.0}},
        'Random Forest': {'val_f1': 0.7024, 'test_f1': 0.6963,
                          'params': {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 5}},
        'XGBoost': {'val_f1': 0.7139, 'test_f1': 0.7328,
                    'params': {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'min_child_weight': 5}}
    }
}
print("Baseline metrics loaded.")

## 1. Data Preparation

Replicate exact preprocessing from notebook 8b.

In [ ]:
# Load multi-route data (with holiday features pre-computed)
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Filter low-volume airline-routes
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print(f"Records after filtering (>={volume_threshold} flights/mo): {len(df_filtered)}")

In [ ]:
# Exclude anomalous routes (same as 8b)
# Reason: Anomalous 2019 data causes unreliable predictions (see notebook 6c)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
records_before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print(f"Excluded anomalous routes: {anomalous_routes}")
print(f"Records before: {records_before}")
print(f"Records after:  {len(df_filtered)}")

In [ ]:
# Feature engineering
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

In [ ]:
# Train/Validation/Test split (same as 8b)
train_mask = (((df_clean['year'] >= 2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2023): {train_mask.sum()} samples")
print(f"Val (2018, 2024):        {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define feature set (same as 8b)
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']
FINAL_FEATURES = base_features + weather_features + holiday_features

# Prepare data
X_train = df_clean.loc[train_mask, FINAL_FEATURES].values
X_val = df_clean.loc[val_mask, FINAL_FEATURES].values
X_test = df_clean.loc[test_mask, FINAL_FEATURES].values

y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Features: {len(FINAL_FEATURES)}")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

## 2. Ridge Regression Tuning

Tune regularization strength (alpha).

In [ ]:
ridge_alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0, 500.0]

ridge_results = []
for alpha in ridge_alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train_reg)
    val_pred = model.predict(X_val_scaled)
    val_r2 = r2_score(y_val_reg, val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val_reg, val_pred))
    ridge_results.append({'alpha': alpha, 'val_r2': val_r2, 'val_rmse': val_rmse})

ridge_df = pd.DataFrame(ridge_results)
best_ridge_idx = ridge_df['val_r2'].idxmax()
best_ridge_alpha = ridge_df.loc[best_ridge_idx, 'alpha']
best_ridge_val_r2 = ridge_df.loc[best_ridge_idx, 'val_r2']

print("Ridge Tuning Results:")
print("-" * 50)
print(f"{'alpha':>10} {'val_r2':>12} {'val_rmse':>12}")
print("-" * 50)
for _, row in ridge_df.iterrows():
    marker = " *" if row['alpha'] == best_ridge_alpha else ""
    print(f"{row['alpha']:>10.2f} {row['val_r2']:>12.4f} {row['val_rmse']:>12.4f}{marker}")

print(f"\nBest alpha: {best_ridge_alpha} (Val R²: {best_ridge_val_r2:.4f})")
print(f"Baseline:   100 (Val R²: {BASELINE_8B['regression']['Ridge']['val_r2']:.4f})")

In [ ]:
# Plot regularization curve
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(ridge_df['alpha'], ridge_df['val_r2'], 'o-', linewidth=2, markersize=8, color='steelblue')
ax.axhline(BASELINE_8B['regression']['Ridge']['val_r2'], color='red', linestyle='--', 
           label=f"Baseline (alpha=100): {BASELINE_8B['regression']['Ridge']['val_r2']:.4f}")

ax.scatter([best_ridge_alpha], [best_ridge_val_r2], color='green', s=150, zorder=5,
           label=f"Best: alpha={best_ridge_alpha}, R²={best_ridge_val_r2:.4f}")

ax.set_xscale('log')
ax.set_xlabel('alpha (regularization strength)')
ax.set_ylabel('Validation R²')
ax.set_title('Ridge Regression: Regularization Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Random Forest Regression Tuning

Tune max_depth and min_samples_leaf.

In [ ]:
rf_max_depths = [5, 8, 10, 15, 20, None]
rf_min_samples_leafs = [1, 3, 5, 10, 20]

rf_reg_results = []
total_combos = len(rf_max_depths) * len(rf_min_samples_leafs)
print(f"Testing {total_combos} combinations...")

for i, (max_depth, min_samples_leaf) in enumerate(itertools.product(rf_max_depths, rf_min_samples_leafs)):
    model = RandomForestRegressor(n_estimators=100, max_depth=max_depth, 
                                   min_samples_leaf=min_samples_leaf, 
                                   random_state=42, n_jobs=-1)
    model.fit(X_train, y_train_reg)
    val_pred = model.predict(X_val)
    val_r2 = r2_score(y_val_reg, val_pred)
    rf_reg_results.append({
        'max_depth': max_depth if max_depth else 'None',
        'min_samples_leaf': min_samples_leaf,
        'val_r2': val_r2
    })
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{total_combos} done...")

rf_reg_df = pd.DataFrame(rf_reg_results)
best_rf_reg_idx = rf_reg_df['val_r2'].idxmax()
best_rf_reg_params = rf_reg_df.loc[best_rf_reg_idx]

print(f"\nBest RF Regression params:")
print(f"  max_depth: {best_rf_reg_params['max_depth']}")
print(f"  min_samples_leaf: {best_rf_reg_params['min_samples_leaf']}")
print(f"  Val R²: {best_rf_reg_params['val_r2']:.4f}")
print(f"\nBaseline Val R²: {BASELINE_8B['regression']['Random Forest']['val_r2']:.4f}")

In [ ]:
# Plot heatmap
rf_reg_df_plot = rf_reg_df.copy()
rf_reg_df_plot['max_depth'] = rf_reg_df_plot['max_depth'].astype(str)
pivot = rf_reg_df_plot.pivot(index='min_samples_leaf', columns='max_depth', values='val_r2')

# Reorder columns
col_order = ['5', '8', '10', '15', '20', 'None']
pivot = pivot[[c for c in col_order if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')

ax.set_xticks(np.arange(len(pivot.columns)))
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('max_depth')
ax.set_ylabel('min_samples_leaf')
ax.set_title('Random Forest Regression: Validation R² Heatmap')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, label='Validation R²')
plt.tight_layout()
plt.show()

## 4. Logistic Regression Tuning

Tune regularization strength (C = 1/alpha).

In [ ]:
logreg_Cs = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0]

logreg_results = []
for C in logreg_Cs:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train_clf)
    val_pred = model.predict(X_val_scaled)
    val_f1 = f1_score(y_val_clf, val_pred)
    logreg_results.append({'C': C, 'val_f1': val_f1})

logreg_df = pd.DataFrame(logreg_results)
best_logreg_idx = logreg_df['val_f1'].idxmax()
best_logreg_C = logreg_df.loc[best_logreg_idx, 'C']
best_logreg_val_f1 = logreg_df.loc[best_logreg_idx, 'val_f1']

print("Logistic Regression Tuning Results:")
print("-" * 40)
print(f"{'C':>10} {'val_f1':>12}")
print("-" * 40)
for _, row in logreg_df.iterrows():
    marker = " *" if row['C'] == best_logreg_C else ""
    print(f"{row['C']:>10.2f} {row['val_f1']:>12.4f}{marker}")

print(f"\nBest C: {best_logreg_C} (Val F1: {best_logreg_val_f1:.4f})")
print(f"Baseline: 1.0 (Val F1: {BASELINE_8B['classification']['Logistic']['val_f1']:.4f})")

In [ ]:
# Plot regularization curve
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(logreg_df['C'], logreg_df['val_f1'], 'o-', linewidth=2, markersize=8, color='steelblue')
ax.axhline(BASELINE_8B['classification']['Logistic']['val_f1'], color='red', linestyle='--',
           label=f"Baseline (C=1.0): {BASELINE_8B['classification']['Logistic']['val_f1']:.4f}")

ax.scatter([best_logreg_C], [best_logreg_val_f1], color='green', s=150, zorder=5,
           label=f"Best: C={best_logreg_C}, F1={best_logreg_val_f1:.4f}")

ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Validation F1')
ax.set_title('Logistic Regression: Regularization Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Random Forest Classification Tuning

In [ ]:
rf_clf_results = []
total_combos = len(rf_max_depths) * len(rf_min_samples_leafs)
print(f"Testing {total_combos} combinations...")

for i, (max_depth, min_samples_leaf) in enumerate(itertools.product(rf_max_depths, rf_min_samples_leafs)):
    model = RandomForestClassifier(n_estimators=100, max_depth=max_depth,
                                    min_samples_leaf=min_samples_leaf,
                                    random_state=42, n_jobs=-1)
    model.fit(X_train, y_train_clf)
    val_pred = model.predict(X_val)
    val_f1 = f1_score(y_val_clf, val_pred)
    rf_clf_results.append({
        'max_depth': max_depth if max_depth else 'None',
        'min_samples_leaf': min_samples_leaf,
        'val_f1': val_f1
    })
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{total_combos} done...")

rf_clf_df = pd.DataFrame(rf_clf_results)
best_rf_clf_idx = rf_clf_df['val_f1'].idxmax()
best_rf_clf_params = rf_clf_df.loc[best_rf_clf_idx]

print(f"\nBest RF Classification params:")
print(f"  max_depth: {best_rf_clf_params['max_depth']}")
print(f"  min_samples_leaf: {best_rf_clf_params['min_samples_leaf']}")
print(f"  Val F1: {best_rf_clf_params['val_f1']:.4f}")
print(f"\nBaseline Val F1: {BASELINE_8B['classification']['Random Forest']['val_f1']:.4f}")

In [ ]:
# Plot heatmap
rf_clf_df_plot = rf_clf_df.copy()
rf_clf_df_plot['max_depth'] = rf_clf_df_plot['max_depth'].astype(str)
pivot_clf = rf_clf_df_plot.pivot(index='min_samples_leaf', columns='max_depth', values='val_f1')
pivot_clf = pivot_clf[[c for c in col_order if c in pivot_clf.columns]]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot_clf.values, cmap='RdYlGn', aspect='auto')

ax.set_xticks(np.arange(len(pivot_clf.columns)))
ax.set_yticks(np.arange(len(pivot_clf.index)))
ax.set_xticklabels(pivot_clf.columns)
ax.set_yticklabels(pivot_clf.index)
ax.set_xlabel('max_depth')
ax.set_ylabel('min_samples_leaf')
ax.set_title('Random Forest Classification: Validation F1 Heatmap')

for i in range(len(pivot_clf.index)):
    for j in range(len(pivot_clf.columns)):
        val = pivot_clf.values[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, label='Validation F1')
plt.tight_layout()
plt.show()

## 6. XGBoost Classification Tuning

In [ ]:
if HAS_XGB:
    xgb_max_depths = [3, 5, 7, 10]
    xgb_learning_rates = [0.01, 0.05, 0.1, 0.2]
    xgb_n_estimators = [50, 100, 200]
    
    xgb_results = []
    total_combos = len(xgb_max_depths) * len(xgb_learning_rates) * len(xgb_n_estimators)
    print(f"Testing {total_combos} combinations...")
    
    for i, (max_depth, lr, n_est) in enumerate(itertools.product(xgb_max_depths, xgb_learning_rates, xgb_n_estimators)):
        model = xgb.XGBClassifier(n_estimators=n_est, max_depth=max_depth, 
                                   learning_rate=lr, min_child_weight=5,
                                   random_state=42, n_jobs=-1)
        model.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        val_pred = model.predict(X_val)
        val_f1 = f1_score(y_val_clf, val_pred)
        xgb_results.append({
            'max_depth': max_depth,
            'learning_rate': lr,
            'n_estimators': n_est,
            'val_f1': val_f1
        })
        if (i+1) % 12 == 0:
            print(f"  {i+1}/{total_combos} done...")
    
    xgb_df = pd.DataFrame(xgb_results)
    best_xgb_idx = xgb_df['val_f1'].idxmax()
    best_xgb_params = xgb_df.loc[best_xgb_idx]
    
    print(f"\nBest XGBoost params:")
    print(f"  max_depth: {best_xgb_params['max_depth']}")
    print(f"  learning_rate: {best_xgb_params['learning_rate']}")
    print(f"  n_estimators: {best_xgb_params['n_estimators']}")
    print(f"  Val F1: {best_xgb_params['val_f1']:.4f}")
    print(f"\nBaseline Val F1: {BASELINE_8B['classification']['XGBoost']['val_f1']:.4f}")
else:
    print("XGBoost not available, skipping.")

In [ ]:
if HAS_XGB:
    # Show top 10 XGBoost configurations
    print("Top 10 XGBoost Configurations:")
    print("-" * 60)
    print(f"{'max_depth':>10} {'lr':>8} {'n_est':>8} {'val_f1':>10}")
    print("-" * 60)
    for _, row in xgb_df.nlargest(10, 'val_f1').iterrows():
        print(f"{row['max_depth']:>10} {row['learning_rate']:>8.2f} {row['n_estimators']:>8} {row['val_f1']:>10.4f}")

## 7. Final Evaluation on Test Set

Train models with tuned parameters and evaluate on held-out test set.

In [ ]:
# Collect best parameters
tuned_params = {
    'Ridge': {'alpha': best_ridge_alpha},
    'RF_Reg': {
        'max_depth': None if best_rf_reg_params['max_depth'] == 'None' else int(best_rf_reg_params['max_depth']),
        'min_samples_leaf': int(best_rf_reg_params['min_samples_leaf'])
    },
    'Logistic': {'C': best_logreg_C},
    'RF_Clf': {
        'max_depth': None if best_rf_clf_params['max_depth'] == 'None' else int(best_rf_clf_params['max_depth']),
        'min_samples_leaf': int(best_rf_clf_params['min_samples_leaf'])
    },
}

if HAS_XGB:
    tuned_params['XGBoost'] = {
        'max_depth': int(best_xgb_params['max_depth']),
        'learning_rate': best_xgb_params['learning_rate'],
        'n_estimators': int(best_xgb_params['n_estimators'])
    }

print("Tuned Parameters:")
for model, params in tuned_params.items():
    print(f"  {model}: {params}")

In [ ]:
# Train and evaluate with tuned parameters
final_results = {'regression': {}, 'classification': {}}

# Ridge
ridge_tuned = Ridge(alpha=tuned_params['Ridge']['alpha'])
ridge_tuned.fit(X_train_scaled, y_train_reg)
ridge_test_pred = ridge_tuned.predict(X_test_scaled)
ridge_test_r2 = r2_score(y_test_reg, ridge_test_pred)
final_results['regression']['Ridge'] = {'test_r2': ridge_test_r2, 'params': tuned_params['Ridge']}

# RF Regression
rf_reg_tuned = RandomForestRegressor(n_estimators=100, 
                                      max_depth=tuned_params['RF_Reg']['max_depth'],
                                      min_samples_leaf=tuned_params['RF_Reg']['min_samples_leaf'],
                                      random_state=42, n_jobs=-1)
rf_reg_tuned.fit(X_train, y_train_reg)
rf_reg_test_pred = rf_reg_tuned.predict(X_test)
rf_reg_test_r2 = r2_score(y_test_reg, rf_reg_test_pred)
final_results['regression']['Random Forest'] = {'test_r2': rf_reg_test_r2, 'params': tuned_params['RF_Reg']}

# Logistic
logreg_tuned = LogisticRegression(C=tuned_params['Logistic']['C'], max_iter=1000, random_state=42)
logreg_tuned.fit(X_train_scaled, y_train_clf)
logreg_test_pred = logreg_tuned.predict(X_test_scaled)
logreg_test_f1 = f1_score(y_test_clf, logreg_test_pred)
final_results['classification']['Logistic'] = {'test_f1': logreg_test_f1, 'params': tuned_params['Logistic']}

# RF Classification
rf_clf_tuned = RandomForestClassifier(n_estimators=100,
                                       max_depth=tuned_params['RF_Clf']['max_depth'],
                                       min_samples_leaf=tuned_params['RF_Clf']['min_samples_leaf'],
                                       random_state=42, n_jobs=-1)
rf_clf_tuned.fit(X_train, y_train_clf)
rf_clf_test_pred = rf_clf_tuned.predict(X_test)
rf_clf_test_f1 = f1_score(y_test_clf, rf_clf_test_pred)
final_results['classification']['Random Forest'] = {'test_f1': rf_clf_test_f1, 'params': tuned_params['RF_Clf']}

# XGBoost
if HAS_XGB:
    xgb_tuned = xgb.XGBClassifier(n_estimators=tuned_params['XGBoost']['n_estimators'],
                                   max_depth=tuned_params['XGBoost']['max_depth'],
                                   learning_rate=tuned_params['XGBoost']['learning_rate'],
                                   min_child_weight=5, random_state=42, n_jobs=-1)
    xgb_tuned.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
    xgb_test_pred = xgb_tuned.predict(X_test)
    xgb_test_f1 = f1_score(y_test_clf, xgb_test_pred)
    final_results['classification']['XGBoost'] = {'test_f1': xgb_test_f1, 'params': tuned_params['XGBoost']}

print("Final models trained.")

In [ ]:
# Summary comparison
print("=" * 90)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 90)

print("\nREGRESSION (Test Set):")
print("-" * 90)
print(f"{'Model':<15} {'Default R²':>12} {'Tuned R²':>12} {'Δ R²':>10} {'Best Parameters'}")
print("-" * 90)

for model in ['Ridge', 'Random Forest']:
    default = BASELINE_8B['regression'][model]['test_r2']
    tuned = final_results['regression'][model]['test_r2']
    delta = tuned - default
    params = final_results['regression'][model]['params']
    params_str = ', '.join([f"{k}={v}" for k, v in params.items()])
    print(f"{model:<15} {default:>12.4f} {tuned:>12.4f} {delta:>+10.4f}   {params_str}")

print("\nCLASSIFICATION (Test Set):")
print("-" * 90)
print(f"{'Model':<15} {'Default F1':>12} {'Tuned F1':>12} {'Δ F1':>10} {'Best Parameters'}")
print("-" * 90)

for model in ['Logistic', 'Random Forest', 'XGBoost']:
    if model not in final_results['classification']:
        continue
    default = BASELINE_8B['classification'][model]['test_f1']
    tuned = final_results['classification'][model]['test_f1']
    delta = tuned - default
    params = final_results['classification'][model]['params']
    params_str = ', '.join([f"{k}={v}" for k, v in params.items()])
    print(f"{model:<15} {default:>12.4f} {tuned:>12.4f} {delta:>+10.4f}   {params_str}")

In [ ]:
# Visualization: Default vs Tuned
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Regression
ax = axes[0]
models_reg = ['Ridge', 'Random Forest']
default_r2 = [BASELINE_8B['regression'][m]['test_r2'] for m in models_reg]
tuned_r2 = [final_results['regression'][m]['test_r2'] for m in models_reg]

x = np.arange(len(models_reg))
width = 0.35
ax.bar(x - width/2, default_r2, width, label='Default', color='#95a5a6', alpha=0.8)
ax.bar(x + width/2, tuned_r2, width, label='Tuned', color='steelblue', alpha=0.8)

ax.set_ylabel('Test R²')
ax.set_title('Regression: Default vs Tuned')
ax.set_xticks(x)
ax.set_xticklabels(models_reg)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add delta annotations
for i, (d, t) in enumerate(zip(default_r2, tuned_r2)):
    delta = t - d
    ax.annotate(f'{delta:+.4f}', xy=(i + width/2, t), xytext=(0, 5),
                textcoords='offset points', ha='center', fontsize=9)

# Classification
ax = axes[1]
models_clf = ['Logistic', 'Random Forest']
if HAS_XGB:
    models_clf.append('XGBoost')

default_f1 = [BASELINE_8B['classification'][m]['test_f1'] for m in models_clf]
tuned_f1 = [final_results['classification'][m]['test_f1'] for m in models_clf]

x = np.arange(len(models_clf))
ax.bar(x - width/2, default_f1, width, label='Default', color='#95a5a6', alpha=0.8)
ax.bar(x + width/2, tuned_f1, width, label='Tuned', color='steelblue', alpha=0.8)

ax.set_ylabel('Test F1')
ax.set_title('Classification: Default vs Tuned')
ax.set_xticks(x)
ax.set_xticklabels(models_clf)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add delta annotations
for i, (d, t) in enumerate(zip(default_f1, tuned_f1)):
    delta = t - d
    ax.annotate(f'{delta:+.4f}', xy=(i + width/2, t), xytext=(0, 5),
                textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()